# Chapter 2 — Exercises

Use this notebook only for the chapter exercises. Attempt each exercise before checking references or solutions.

In [77]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "exercises" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

pandas: 3.0.6
Matplotlib: 3.11.2
scikit-learn: 1.9.1


## Core: data loading, preprocessing, models, cross-validation

Reused from `chapters/02-end-to-end-project.ipynb` as a starting point for the exercises below.

In [78]:
import tarfile
import urllib.request

def load_housing():
    tarball_path = PROJECT_ROOT / "datasets" / "housing.tgz"
    if not tarball_path.is_file():
        tarball_path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as tar:
            tar.extractall(path=tarball_path.parent, filter="data")
    return pd.read_csv(tarball_path.parent / "housing" / "housing.csv")

housing_full = load_housing()


In [79]:
from sklearn.model_selection import train_test_split

# Stratified split on income_cat so the test set represents the full income distribution.
housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5],
)

strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2, stratify=housing_full["income_cat"], random_state=RANDOM_STATE
)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()


In [80]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.k_means_ = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        self.k_means_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.k_means_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]


def column_ratio(X):
    return X[:, [0]] / X[:, [1]]


def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]


def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler(),
    )


def log_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(np.log, feature_names_out="one-to-one"),
        StandardScaler(),
    )


cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=RANDOM_STATE)

default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline(), ["total_bedrooms", "total_rooms", "population", "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include="object")),
], remainder=default_num_pipeline)


In [81]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor

# Linear Regression baseline
lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)
lin_rmse = root_mean_squared_error(housing_labels, lin_reg.predict(housing))
print(f"Linear Regression train RMSE: {lin_rmse:,.2f}")  # underfits

# Decision Tree
tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=RANDOM_STATE))
tree_reg.fit(housing, housing_labels)
tree_rmse = root_mean_squared_error(housing_labels, tree_reg.predict(housing))
print(f"Decision Tree train RMSE: {tree_rmse:,.2f}")  # overfits (0 error on training set)

# Cross-validation to get a realistic estimate of generalization error
tree_rmses = -cross_val_score(
    tree_reg, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10
)
print("Decision Tree CV RMSE:")
print(pd.Series(tree_rmses).describe())

# Random Forest
forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=RANDOM_STATE))
forest_rmses = -cross_val_score(
    forest_reg, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10
)
print("Random Forest CV RMSE:")
print(pd.Series(forest_rmses).describe())


Linear Regression train RMSE: 68,972.89
Decision Tree train RMSE: 0.00
Decision Tree CV RMSE:
count       10.000000
mean     67013.360949
std       1460.198570
min      64289.376198
25%      66776.146282
50%      67086.216281
75%      68140.275029
max      68659.294290
dtype: float64
Random Forest CV RMSE:
count       10.000000
mean     47124.604437
std       1069.311372
min      45292.329302
25%      46712.106520
50%      47172.209883
75%      47561.377695
max      49354.705514
dtype: float64


In [82]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from scipy.stats import randint

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=RANDOM_STATE)),
])

param_distribs = {
    "preprocessing__geo__n_clusters": randint(3, 50),
    "random_forest__max_features": randint(2, 20),
}

rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distribs,
    n_iter=10,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
)
rnd_search.fit(housing, housing_labels)

final_model = rnd_search.best_estimator_
feature_importances = final_model["random_forest"].feature_importances_
print(sorted(zip(feature_importances.round(3), final_model["preprocessing"].get_feature_names_out()), reverse=True))


[(np.float64(0.188), 'log__median_income'), (np.float64(0.078), 'cat__ocean_proximity_INLAND'), (np.float64(0.061), 'bedrooms__ratio'), (np.float64(0.058), 'rooms_per_house__ratio'), (np.float64(0.046), 'people_per_house__ratio'), (np.float64(0.042), 'geo__Cluster 30 similarity'), (np.float64(0.025), 'geo__Cluster 9 similarity'), (np.float64(0.024), 'geo__Cluster 36 similarity'), (np.float64(0.021), 'geo__Cluster 18 similarity'), (np.float64(0.019), 'geo__Cluster 32 similarity'), (np.float64(0.019), 'geo__Cluster 3 similarity'), (np.float64(0.018), 'geo__Cluster 25 similarity'), (np.float64(0.016), 'geo__Cluster 26 similarity'), (np.float64(0.016), 'geo__Cluster 1 similarity'), (np.float64(0.014), 'geo__Cluster 7 similarity'), (np.float64(0.014), 'geo__Cluster 10 similarity'), (np.float64(0.013), 'geo__Cluster 34 similarity'), (np.float64(0.013), 'geo__Cluster 19 similarity'), (np.float64(0.013), 'geo__Cluster 12 similarity'), (np.float64(0.013), 'geo__Cluster 0 similarity'), (np.float

In [83]:
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

final_predictions = final_model.predict(X_test)
final_rmse = root_mean_squared_error(y_test, final_predictions)
print(f"Final model test RMSE: {final_rmse:,.2f}")


Final model test RMSE: 41,423.08


### Exercise 1

Try SVM regressor with different hyperparameters.

In [84]:
from sklearn.svm import SVR

svr_pipeline = Pipeline([
  ("preprocessing", preprocessing),
  ("svr", SVR())
])

params = [
  {'svr__kernel': ['linear'], 'svr__C': [10., 30., 100., 300., 1000.,                                             3000., 10000., 30000.0]},
  {'svr__kernel': ['rbf'], 'svr__C': [1.0, 3.0, 10., 30., 100., 300.,1000.0],
 'svr__gamma': [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]},
]

grid_search = GridSearchCV(svr_pipeline, params, cv=3, scoring="neg_root_mean_squared_error")
grid_search.fit(housing.iloc[:5000], housing_labels.iloc[:5000])
svr_grid_search_rmse = -grid_search.best_score_
svr_grid_search_rmse

np.float64(70059.92773220727)

Trying with a randomized search cv

In [85]:
from scipy.stats import expon, loguniform

params = {
  "svr__kernel": ["linear", "rbf"],
  "svr__C": loguniform(20, 200_000),
  "svr__gamma": expon(scale=0.1)
}
randomized_search_svr = RandomizedSearchCV(svr_pipeline, params, cv=3, scoring="neg_root_mean_squared_error", n_iter=10, random_state=42)
randomized_search_svr.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

rs_svr = -randomized_search_svr.best_score_
print(rs_svr)

57311.06310654309


In [86]:
from sklearn.feature_selection import SelectFromModel


optimized_pipeline = Pipeline([
  ("preprocessing", preprocessing),
  ("select_from_model", SelectFromModel(RandomForestRegressor(random_state=42), threshold=0.005)),
  ("svr", SVR(kernel=randomized_search_svr.best_params_["svr__kernel"], C=randomized_search_svr.best_params_["svr__C"], gamma=randomized_search_svr.best_params_["svr__gamma"]))
])

eval = -cross_val_score(optimized_pipeline, housing.iloc[:5000], housing_labels.iloc[:5000], cv=3, scoring="neg_root_mean_squared_error")
eval

array([60010.56543488, 57711.39113174, 55968.93553249])

Implement KMeans

In [144]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_predict

class FeatureFromRegressor(BaseEstimator, TransformerMixin):
  def __init__(self, regressor, target_features):
    self.regressor = regressor
    self.target_features = target_features

  def fit(self, X, y=None):
    if hasattr(X, "columns"):
      self.features_names_in_ = list(X.columns)
      X_df = X
    else:
      X_df = pd.DataFrame(X)

    self.input_features_ = [c for c in self.features_names_in_ if c not in self.target_features]
    self.regressor_ = clone(self.regressor)
    self.regressor_.fit(X_df[self.input_features_], X_df[self.target_features])
    return self

  def transform(self, X, y=None):
    columns = X.columns if hasattr(X, "columns") else None
    X_df = pd.DataFrame(X, columns=columns)
    predictions = self.regressor_.predict(X_df[self.input_features_])
    if predictions.ndim == 1:
      predictions = predictions.reshape(-1, 1)

    # We want to create a pandas data frame of the predictions with the target feature names
    column_names = [f"prediction_{t}" for t in self.target_features]
    preds_df = pd.DataFrame(predictions, columns=column_names, index=X_df.index)
    return pd.concat([X_df, preds_df], axis=1)

  def get_feature_names_out(self, input_features=None):
    return self.features_names_in_ + [f"prediction_{t}" for t in self.target_features]

  def fit_transform(self, X, y=None):
    self.fit(X, y)
    X_df = pd.DataFrame(X)
    preds = cross_val_predict(self.regressor_, X_df[self.input_features_], X_df[self.target_features], cv=5)
    if preds.ndim == 1:
      preds = preds.reshape(-1, 1)
    column_names = [f"prediction_{t}" for t in self.target_features]
    preds_df = pd.DataFrame(preds, columns=column_names, index=X_df.index)
    return pd.concat([X_df, preds_df], axis=1)

In [145]:
from sklearn.neighbors import KNeighborsRegressor

knn_reg = KNeighborsRegressor(n_neighbors=3)
knn_transformer = FeatureFromRegressor(regressor=knn_reg, target_features=["median_income"])
geo_features = housing[["latitude", "longitude", "median_income"]]
results = knn_transformer.fit_transform(geo_features)

In [155]:
transformers = [(name, clone(transformer), columns) for name, transformer, columns in preprocessing.transformers]

# replace the KMeans with the new custom transformer
geo_index = [name for name, _, _ in transformers].index("geo")
transformers[geo_index] = ("geo", clone(knn_transformer), ["latitude", "longitude", "median_income"])
new_geo_preprocessing = ColumnTransformer(transformers, remainder=clone(preprocessing.remainder))

In [161]:
# Run this with the best SVM parameters

new_geo_pipeline = Pipeline([
  ("preprocessing", new_geo_preprocessing),
  ("svr", SVR(kernel=randomized_search_svr.best_params_["svr__kernel"],
              C=randomized_search_svr.best_params_["svr__C"],
              gamma=randomized_search_svr.best_params_["svr__gamma"]))
])

rmse = -cross_val_score(new_geo_pipeline, housing.iloc[:5000], housing_labels.iloc[:5000], scoring="neg_root_mean_squared_error", cv=3).mean()
print(rmse)

54258.30800300307


In [ ]:
# Exploring preparing techniques with RandomSearchCV. This means that we need to find optimal hyperparameters for our Regressor.

geo_params = {
  "preprocessing__geo__regressor__n_neighbors": range(1, 30),
  "preprocessing__geo__regressor__weights": ["uniform", "distance"],
  "svr__C": loguniform(20, 200_000),
  "svr__gamma": expon(scale=0.1)
}

geo_search = RandomizedSearchCV(new_geo_pipeline, geo_params, n_iter=50, cv=3, scoring="neg_root_mean_squared_error", random_state=42)
geo_search.fit(housing.iloc[:5000], housing_labels.iloc[:5000])
print(geo_search.best_params_)
print(-geo_search.best_score_)

{'preprocessing__geo__regressor__n_neighbors': 26, 'preprocessing__geo__regressor__weights': 'uniform', 'svr__C': np.float64(76806.00838197708), 'svr__gamma': np.float64(0.03827307164183185)}


In [169]:
from sklearn.utils.validation import validate_data, check_is_fitted

class StandardScalerClone(BaseEstimator, TransformerMixin):
  def __init__(self, with_mean=True):
    self.with_mean = with_mean

  def fit(self, X, y=None):
    X = validate_data(X, reset=True, ensure_2d=True)
    self.n_features_in = X.shape[1]
    if self.with_mean:
      self.mean_ = X.mean(axis=0)
    self.scale_ = X.std(axis=0, ddof=0)
    return self

  def transform(self, X):
    check_is_fitted(self, ['scale_'])
    X = validate_data(X, ensure_2d=True, reset=False)
    return (X - self.mean_) / self.scale_

  def inverse_transform(self, X):
    check_is_fitted(self)
    X = validate_data(X, ensure_2d=True, reset=False)
    X = X * self.scale_
    if self.with_mean:
      X = X + self.mean_
    return X

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      return getattr(self, "feature_names_in_",
        [f"x{i}" for i in range(self.n_features_in_)])
    else:
      if len(input_features) != self.n_features_in_:
        return ValueError(f"Expected {self.n_features_in_} input features, got {len(input_features)}.")
      if hasattr(self, "feature_names_in_") and not np.all(
                self.feature_names_in_ == input_features
            ):
        raise ValueError("input_features ≠ feature_names_in_")
      return input_features